# Video Preprocess Demo

이 notebook 은 `video-preprocess-workbench` 를 Jupyter 에서 사용하는 최소 예제다.

권장 순서:
1. config 로드
2. 입력 경로 / scan depth override
3. inventory / preview 확인
4. dry-run 또는 실제 batch 실행


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('project root not found')

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from video_preprocess_workbench import load_config, inspect_inputs, save_preview, run_batch
from video_preprocess_workbench.pipeline import apply_overrides

print('project_root =', project_root)


In [ ]:
config_path = project_root / 'configs' / 'example_batch.json'
cfg = load_config(config_path)

cfg = apply_overrides(
    cfg,
    input_path='/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260413_배테스트용_영상들',
    output_dir=str(project_root / 'artifacts' / 'runs'),
    scan_depth=3,
)

cfg.transform.resize_mode = 'fit_pad'
cfg.transform.resize_width = 1280
cfg.transform.resize_height = 720
cfg.transform.target_fps = 10.0

cfg.roi.enabled = False
cfg.preview.video_index = 0
cfg.preview.debug_frame_index = 0

cfg.to_dict()


In [ ]:
inspection = inspect_inputs(cfg)
inspection['summary']


In [ ]:
preview = save_preview(cfg, run_dir=inspection['run_dir'])
preview['preview_info']


In [ ]:
summary = run_batch(cfg, dry_run=True, run_dir=inspection['run_dir'])
summary
